# Topic Analysis of Consumer Complaints — CFPB Student Loan Narratives

**Course:** Project: Data Analysis (DLBDSEDA02) — Development Phase

**Goal.** Analyze a collection of unstructured complaint texts and extract the most
frequently addressed topics. As planned in the conception phase, the corpus consists of
consumer complaints about student loans submitted to the U.S. Consumer Financial
Protection Bureau (CFPB)

**Pipeline** (each step follows the plan approved in the conception phase):
1. Load the data (CFPB complaint narratives, July 2025 – June 2026)
2. Preprocess the narratives into "clean text"
3. Vectorize the texts with two techniques: Bag of Words and TF-IDF
4. Extract topics with two semantic analysis techniques: LSA and LDA
5. Validate and discuss the results

**Libraries**: `pandas`, `re`, `nltk`, `scikit-learn`.

## 1. Setup

Import the libraries and download the NLTK resources needed for tokenization
(`punkt`, `punkt_tab`), stop word removal (`stopwords`) and lemmatization
(`wordnet`, `omw-1.4`). The downloads only happen once; afterwards NLTK uses
the local copy.

In [1]:
import re
from pathlib import Path

import pandas as pd
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import TruncatedSVD, LatentDirichletAllocation

for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4"]:
    nltk.download(pkg, quiet=True)

print("Setup complete.")

Setup complete.


### Configuration

All tunable settings are collected here so experiments only require changing one cell.

In [18]:
DATA_FILE = Path("data/complaints.csv")

# Official CFPB complaint-search API (CSV export). With format=csv the API returns
# ALL records matching the filters. Filters match the conception phase
API_URL = (
    "https://www.consumerfinance.gov/data-research/consumer-complaints/search/api/v1/"
    "?format=csv"
    "&no_aggs=true"
    "&product=Student%20loan"
    "&has_narrative=true"
    "&date_received_min=2025-07-01"
    "&date_received_max=2026-06-30"
)

TEXT_COL = "Consumer complaint narrative"  # column with the unstructured text
LABEL_COL = "Issue"                        # pre-labeled category, used for validation
DATE_COL = "Date received"

N_TOPICS = 8          # number of topics for LSA and LDA
N_TOP_WORDS = 10      # words shown per topic
MIN_DF = 5            # ignore terms in fewer than 5 documents
MAX_DF = 0.95         # ignore terms in more than 95% of documents
RANDOM_STATE = 42     # fixed seed so results are reproducible

## 2. Data loading

If a local copy exists in `data/complaints.csv` it is used (recommended, so results
stay identical between runs). Otherwise the dataset is downloaded once from the
official CFPB API and saved locally. Because complaint narratives are published with
a delay, a fresh download may contain slightly more records than the 6,475 available
at the time of the conception phase.

In [3]:
if DATA_FILE.exists():
    df = pd.read_csv(DATA_FILE)
    print(f"Loaded local file {DATA_FILE} -> {len(df)} rows")
else:
    print("No local file found - downloading from the CFPB API (may take a minute)...")
    try:
        df = pd.read_csv(API_URL)
        DATA_FILE.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(DATA_FILE, index=False)
        print(f"Downloaded {len(df)} rows and saved to {DATA_FILE}")
    except Exception as err:
        print(
            "Automatic download failed. Manual alternative:\n"
            "  1. Open https://www.consumerfinance.gov/data-research/consumer-complaints/search/\n"
            "  2. Filter: product 'Student loan', 'Only show complaints with narratives',\n"
            "     date received 07/01/2025 - 06/30/2026\n"
            "  3. Export as CSV and save it as data/complaints.csv, then re-run this cell."
        )
        raise err

Loaded local file data\complaints.csv -> 6475 rows


In [4]:
# The API CSV export uses human-readable headers ("Consumer complaint narrative").
# The bulk download uses snake_case ("complaint_what_happened"). Normalize so the
# notebook works with either file.
RENAME = {
    "complaint_what_happened": "Consumer complaint narrative",
    "issue": "Issue",
    "date_received": "Date received",
    "product": "Product",
    "sub_product": "Sub-product",
}
df = df.rename(columns={k: v for k, v in RENAME.items() if k in df.columns})

# Keep only rows that actually contain a narrative, and drop duplicated narratives
# (identical template texts submitted multiple times would otherwise dominate topics).
n0 = len(df)
df = df[df[TEXT_COL].notna()]
df = df.drop_duplicates(subset=[TEXT_COL]).reset_index(drop=True)
print(f"{n0} rows loaded -> {len(df)} rows with a unique, non-empty narrative")

dates = pd.to_datetime(df[DATE_COL], errors="coerce")
print(f"Date range: {dates.min().date()} to {dates.max().date()}")

6475 rows loaded -> 6465 rows with a unique, non-empty narrative
Date range: 2025-07-08 to 2026-06-17


In [20]:
# A first look at the data: the pre-labeled issue categories
df[LABEL_COL].value_counts().head(10)

Issue
Dealing with your lender or servicer                               3804
Struggling to repay your loan                                      1347
Incorrect information on your report                                615
Problem with a company's investigation into an existing problem     267
Getting a loan                                                      163
Improper use of your report                                         119
Credit monitoring or identity theft protection services              39
Issue where my lender is my school                                   39
Issue with income share agreement                                    30
Problem with fraud alerts or security freezes                        25
Name: count, dtype: int64

In [19]:
# one raw example narrative (note the 'XXXX' redaction placeholders).
print(df[TEXT_COL].iloc[0][:600])

I sent the below letter to MOHELA requesting information and the correction of my account : MOHELA XXXX XXXX XXXX XXXX, MO, XXXX Subject : Request for Account Correction and Dispute of Delinquency Dear Sir or Madam, I am writing to formally notify you of an error regarding my student loan account, ACCT NUMBER : XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX  XXXX. I have recently received correspondence indicating that my account is reported as delinquent. See Attachment XXXX. However, I have made payments totaling XXXX  XXXX XXXX XXXX XXXX XXXX  which, per my records, were not credited to 


## 3. Preprocessing

The raw narratives are converted into "clean text" in five steps:

1. **Lowercasing** - so that "Loan" and "loan" count as the same word.
2. **Removing punctuation and digits** with a regular expression (`re`); this also
   breaks apart redacted dates like `XX/XX/XXXX`.
3. **Removing the CFPB redaction placeholders** - runs of two or more `x`
   (`xx`, `xxxx`, ...) are artifacts of anonymization, not real words.
4. **Tokenization** with `nltk.word_tokenize` - splitting each text into words.
5. **Stop word removal and lemmatization** - very frequent function words
   ("the", "and", "my") carry no topical meaning and are dropped, together with
   tokens of one or two characters; the remaining tokens are reduced to their base
   form with the WordNet lemmatizer ("payments" -> "payment", "loans" -> "loan"),
   which merges inflected variants of the same word.

In [23]:
STOP_WORDS = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()


def preprocess(text: str) -> list[str]:
    """Turn one raw narrative into a list of clean tokens."""
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)      # keep letters only (removes digits/punctuation)
    text = re.sub(r"\bx{2,}\b", " ", text)     # remove CFPB redaction placeholders (xx, xxxx, ...)
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in STOP_WORDS and len(t) > 2]
    return [lemmatizer.lemmatize(t) for t in tokens]


# Apply to every narrative
df["tokens"] = df[TEXT_COL].apply(preprocess)
df["clean_text"] = df["tokens"].apply(" ".join)

# Discard narratives that are too short to carry a topic after cleaning
before = len(df)
df = df[df["tokens"].apply(len) >= 5].reset_index(drop=True)
print(f"Removed {before - len(df)} very short -> {len(df)} remain")

total_tokens = df["tokens"].apply(len).sum()
print(f"Corpus size after cleaning: {total_tokens} tokens, "
      f"median {int(df['tokens'].apply(len).median())} tokens")

Removed 0 very short -> 6454 remain
Corpus size after cleaning: 728303 tokens, median 89 tokens


In [24]:
# Before/after example of the preprocessing:
print("RAW:   ", df[TEXT_COL].iloc[0][:300], "...")
print()
print("CLEAN: ", df["clean_text"].iloc[0][:300], "...")

RAW:    I sent the below letter to MOHELA requesting information and the correction of my account : MOHELA XXXX XXXX XXXX XXXX, MO, XXXX Subject : Request for Account Correction and Dispute of Delinquency Dear Sir or Madam, I am writing to formally notify you of an error regarding my student loan account, A ...

CLEAN:  sent letter mohela requesting information correction account mohela subject request account correction dispute delinquency dear sir madam writing formally notify error regarding student loan account acct number recently received correspondence indicating account reported delinquent see attachment ho ...


## 4. Vectorization: Bag of Words vs. TF-IDF

Machine learning models need numbers, not words, so each narrative is converted into a
numeric vector over the corpus vocabulary using two techniques:

* **Bag of Words (BoW)** - `CountVectorizer` counts how often each vocabulary term
  occurs in each narrative. Simple and interpretable, but common words get high
  values in every narrative.
* **TF-IDF** - `TfidfVectorizer` weights each term frequency by its inverse narrative
  frequency, so words that are distinctive for a narrative score higher than words
  that appear everywhere in the corpus.

Both use the same vocabulary settings (`min_df=5`, `max_df=0.95`), which drops very
rare terms (mostly typos) and near-ubiquitous terms.

In [9]:
count_vec = CountVectorizer(min_df=MIN_DF, max_df=MAX_DF)
X_bow = count_vec.fit_transform(df["clean_text"])

tfidf_vec = TfidfVectorizer(min_df=MIN_DF, max_df=MAX_DF)
X_tfidf = tfidf_vec.fit_transform(df["clean_text"])

sparsity = 100 * (1 - X_bow.nnz / (X_bow.shape[0] * X_bow.shape[1]))
print(f"Document-term matrix: {X_bow.shape[0]} documents x {X_bow.shape[1]} terms")
print(f"Sparsity: {sparsity:.1f}% of matrix entries are zero")

Document-term matrix: 6454 documents x 4527 terms
Sparsity: 98.4% of matrix entries are zero


In [10]:
# Comparison of the two representations: the highest-weighted terms overall.
feature_names = count_vec.get_feature_names_out()

bow_totals = X_bow.sum(axis=0).A1
tfidf_totals = X_tfidf.sum(axis=0).A1

comparison = pd.DataFrame({
    "Top terms by raw count (BoW)": [feature_names[i] for i in bow_totals.argsort()[::-1][:15]],
    "Top terms by TF-IDF weight":   [feature_names[i] for i in tfidf_totals.argsort()[::-1][:15]],
})
comparison.index = range(1, 16)
comparison

,Top terms by raw count (BoW),Top terms by TF-IDF weight
1,loan,loan
2,payment,payment
3,account,mohela
4,credit,account
5,student,credit
6,mohela,student
7,year,year
8,time,interest
9,interest,reporting
10,information,information


**Brief comparison.** Both representations use the same vocabulary, so the matrices
have identical dimensions. The difference is the weighting: BoW ranks terms purely by
how often they occur, so generic corpus-wide words dominate the top of the list.
TF-IDF pushes terms upward that are frequent in *some* narratives but not in *all* of
them, which makes it better suited for distinguishing narratives from one another.
Consequently the two techniques are paired with the topic models as follows: LSA is
applied to the TF-IDF matrix (it benefits from the discriminative weighting), while
LDA is applied to the BoW counts (as a probabilistic model of word counts, LDA
expects integer frequencies, not weighted values).

## 5. Topic extraction

### 5.1 Latent Semantic Analysis (LSA)

LSA applies a truncated singular value decomposition (SVD) to the TF-IDF
narrative-term matrix. The decomposition finds directions ("components") in which the
narrative vary most; words that load strongly on the same component tend to co-occur
and can be read as one latent topic. Note that SVD components may contain negative
loadings and are sorted by explained variance, so the first components capture the
broadest patterns.

In [11]:
def top_words_table(components, feature_names, n_top_words, prefix):
    """Return a DataFrame with the highest-loading words of each topic/component."""
    rows = {}
    for i, comp in enumerate(components):
        top_idx = comp.argsort()[::-1][:n_top_words]
        rows[f"{prefix} {i}"] = [feature_names[j] for j in top_idx]
    table = pd.DataFrame(rows).T
    table.columns = [f"word {j + 1}" for j in range(n_top_words)]
    return table


lsa = TruncatedSVD(n_components=N_TOPICS, random_state=RANDOM_STATE)
lsa_doc_topic = lsa.fit_transform(X_tfidf)

print(f"Explained variance of the {N_TOPICS} components: "
      f"{lsa.explained_variance_ratio_.sum():.1%}")
lsa_topics = top_words_table(lsa.components_, tfidf_vec.get_feature_names_out(),
                             N_TOP_WORDS, "LSA topic")
lsa_topics

Explained variance of the 8 components: 8.7%


,word 1,word 2,word 3,word 4,word 5,word 6,word 7,word 8,word 9,word 10
LSA topic 0,loan,payment,mohela,account,credit,student,interest,year,reporting,forbearance
LSA topic 1,reporting,credit,inaccurate,account,nelnet,report,reported,late,fcra,act
LSA topic 2,payment,late,account,reporting,delinquency,month,forbearance,qualifying,pslf,reported
LSA topic 3,pslf,qualifying,forgiveness,repayment,application,forbearance,request,idr,buyback,count
LSA topic 4,interest,balance,repayment,principal,payment,forbearance,plan,rate,accrued,loan
LSA topic 5,mohela,refund,interest,balance,account,treasury,principal,overpayment,accrued,mohelas
LSA topic 6,payment,refund,sallie,mae,made,late,climb,history,private,balance
LSA topic 7,sallie,mae,application,hardship,call,repayment,option,refund,financial,plan


### 5.2 Latent Dirichlet Allocation (LDA)

LDA is a probabilistic generative model: every narrative is a mixture of topics, and
every topic is a probability distribution over words. Unlike LSA it produces
non-negative, directly interpretable word probabilities. It is fitted on the BoW
count matrix.

In [12]:
lda = LatentDirichletAllocation(
    n_components=N_TOPICS,
    learning_method="batch",
    max_iter=20,
    random_state=RANDOM_STATE,
)
lda_doc_topic = lda.fit_transform(X_bow)

lda_topics = top_words_table(lda.components_, count_vec.get_feature_names_out(),
                             N_TOP_WORDS, "LDA topic")
lda_topics

,word 1,word 2,word 3,word 4,word 5,word 6,word 7,word 8,word 9,word 10
LDA topic 0,loan,student,time,get,year,payment,told,would,pay,never
LDA topic 1,interest,loan,balance,payment,sallie,mae,amount,principal,rate,forbearance
LDA topic 2,mohela,payment,application,forbearance,loan,month,refund,request,year,received
LDA topic 3,information,loan,account,credit,debt,reporting,act,dispute,violation,collection
LDA topic 4,payment,account,mohela,due,email,issue,call,received,made,loan
LDA topic 5,credit,reporting,account,payment,loan,nelnet,reported,late,delinquency,report
LDA topic 6,loan,school,student,discharge,program,financial,private,navient,federal,misconduct
LDA topic 7,loan,payment,repayment,federal,borrower,student,forgiveness,request,mohela,account


In [25]:
# Assign each narrative its dominant LDA topic and check how the corpus is distributed.
df["lda_topic"] = lda_doc_topic.argmax(axis=1)
df["lda_topic"].value_counts().sort_index().rename("number of complaints").to_frame()

,number of complaints
lda_topic,
0,1496
1,466
2,880
3,685
4,662
5,800
6,580
7,885


### 5.3 Validation against the pre-labeled issue categories

The CFPB dataset ships a pre-labeled `Issue` category for every complaint. The
topics were learned *without* seeing these labels, so a cross-tabulation of the
dominant LDA topic against `Issue` is an external check: if a topic's narratives
concentrate in one issue category, the topic captures a real, human-recognized theme.

In [14]:
crosstab = pd.crosstab(df["lda_topic"], df[LABEL_COL], normalize="index")

validation = pd.DataFrame(index=range(N_TOPICS))
validation.index.name = "LDA topic"
validation["top words"] = [
    " ".join(lda_topics.loc[f"LDA topic {i}"].iloc[:5]) for i in range(N_TOPICS)
]
# Aligning on the index keeps this correct even if a topic is never dominant.
validation["complaints"] = df["lda_topic"].value_counts().reindex(range(N_TOPICS), fill_value=0)
validation["most frequent Issue label"] = crosstab.idxmax(axis=1)
validation["share of topic"] = crosstab.max(axis=1).round(2)
validation

,top words,complaints,most frequent Issue label,share of topic
LDA topic,,,,
0,loan student time get year,1496,Dealing with your lender or servicer,0.56
1,interest loan balance payment sallie,466,Dealing with your lender or servicer,0.69
2,mohela payment application forbearance loan,880,Dealing with your lender or servicer,0.73
3,information loan account credit debt,685,Dealing with your lender or servicer,0.49
4,payment account mohela due email,662,Dealing with your lender or servicer,0.86
5,credit reporting account payment loan,800,Incorrect information on your report,0.43
6,loan school student discharge program,580,Dealing with your lender or servicer,0.52
7,loan payment repayment federal borrower,885,Dealing with your lender or servicer,0.66


### 5.4 Sensitivity to the number of topics

The number of topics K is a free parameter. This cell compares the LDA output
for a smaller and a larger K to judge which choice is most interpretable.

In [16]:
for k in [5, 12]:
    lda_k = LatentDirichletAllocation(n_components=k, learning_method="batch", max_iter=20, random_state=RANDOM_STATE)
    lda_k.fit(X_bow)
    print(f"\n===== LDA with K={k} =====")
    print(top_words_table(lda_k.components_, count_vec.get_feature_names_out(), N_TOP_WORDS, "topic"))


===== LDA with K=12 =====
               word 1       word 2     word 3       word 4       word 5  \
topic 0          loan      payment       time          get         year   
topic 1        sallie          mae       loan     document    deferment   
topic 2        mohela  application    request  forbearance    submitted   
topic 3       account  information     credit    reporting         debt   
topic 4       payment      account     mohela          due         loan   
topic 5        credit    reporting    payment      account         loan   
topic 6          loan       school    program      private        climb   
topic 7       payment         loan  repayment         plan  forgiveness   
topic 8   application    complaint    request      federal         loan   
topic 9          loan    discharge    student   department    education   
topic 10         loan      student     school         debt      college   
topic 11     interest         loan    balance    principal  forbearance  

## 6. Discussion of the results

**Identified topics.** The eight LDA topics can be interpreted as recognizable complaint themes:

| Topic | Interpretation | Complaints |
|---|---|---|
| 0 | General servicer communication — unresolved requests and repeated contact ("told", "would", "never") | 1,496 |
| 1 | Interest accrual and growing loan balances (incl. Sallie Mae) | 466 |
| 2 | Processing of applications, forbearance requests and refunds, mainly MOHELA | 880 |
| 3 | Credit-report disputes, collection and Fair Credit Reporting Act violations | 685 |
| 4 | Payment posting and billing problems, contact by e-mail and phone | 662 |
| 5 | Incorrect late/delinquency marks on credit reports (incl. Nelnet) | 800 |
| 6 | School-related discharge and misconduct, incl. Navient and private lenders | 580 |
| 7 | Repayment plans and loan forgiveness (PSLF, IDR) | 885 |

The most prevalent specific themes are repayment plans and forgiveness (topic 7),
application/forbearance processing (topic 2), and credit reporting, which appears as two
distinct facets (topics 3 and 5, together 1,485 complaints). Topic 0 is the largest but
least specific: it absorbs the generic narrative vocabulary shared by most complaints,
a known behavior of LDA on homogeneous corpora.

**BoW vs. TF-IDF.** The two top-15 term lists overlap strongly because the corpus is
topically homogeneous, every narrative concerns student loans. The re-weighting is
nevertheless visible: TF-IDF moves the servicer name "mohela" from rank 6 to rank 3 and
promotes situation-specific terms such as "forbearance" and "balance", while the broadly
used "federal" drops out of the TF-IDF top 15. TF-IDF thus emphasizes exactly the
distinctive vocabulary that later separates the topics.

**LSA vs. LDA.** Both techniques recover the same core themes (credit reporting,
PSLF/forgiveness, interest and balances, MOHELA processing, Sallie Mae/private loans),
which increases confidence in the results. The LDA output is easier to interpret: its
topics are non-negative word distributions, whereas the first LSA component mainly
reproduces the overall corpus vocabulary and component 2 mixes delinquency reporting
with PSLF terms. The eight LSA components explain only 8.7% of the TF-IDF variance,
which is typical for very sparse text matrices. LDA is therefore used as the primary
result.

**Validation.** Seven of the eight topics have "Dealing with your lender or servicer"
as their most frequent Issue label, unsurprising, since this label alone covers 59% of
all complaints. The informative finding is that the topics *subdivide* this broad label
into distinct operational themes (billing, application processing, interest,
forgiveness); topic 4 is the most concentrated (86% of its complaints under one label).
Topic 5 instead aligns with "Incorrect information on your report", and the lower
concentration of topic 3 (49%) matches the fact that dispute and collection issues are
spread over several smaller labels. The unsupervised topics are thus consistent with,
and finer-grained than, the pre-labeled categories.

**Number of topics.** With K = 12 the themes split further while remaining
interpretable, for example, loan discharge and school misconduct separate into topics
of their own. K = 8 is kept as a balance between coverage and redundancy.

**Limitations.** The WordNet lemmatizer is applied without part-of-speech tags, so some
word forms remain unmerged; the number of topics is chosen manually; the "XXXX"
redactions remove names and dates that could carry information; and the broad topic 0
could be reduced with additional domain-specific stop words (e.g. "loan", "student").
These points can be revisited in the finalization phase.

## 7. Reproducibility

Fixed random seeds (`random_state=42`) are used for LSA and LDA. The exact library
versions of this run:

In [17]:
import sklearn
print("pandas       ", pd.__version__)
print("nltk         ", nltk.__version__)
print("scikit-learn ", sklearn.__version__)

pandas        3.0.5
nltk          3.10.3
scikit-learn  1.9.0
